## Feature Engineering Espaciotemporal 

**Nota Metodológica:** Los datos climáticos continuos (Rasters) utilizados en este notebook fueron procesados y extraídos asíncronamente utilizando los servidores de Google Earth Engine. 
[🌐 Ver Script de Extracción en Google Colab](https://colab.research.google.com/drive/1yhUlnHcjPREOiRYKhn1yl2HjsQm3c1Pu#scrollTo=vN_G1BTzZT-d)

## 1. Justificación del Paradigma

El modelado epidemiológico inicial mediante un enfoque tabular (CSV) presenta limitaciones para evaluar el riesgo en zonas geográficas sin reportes previos. Para solucionar este problema (falta de datos sobre dónde la mosca *no* puede sobrevivir), hemos optado por utilizar la arquitectura hacia el **Modelado de Nicho Ecológico (ENM)** utilizando la máxima entropía (MaxEnt).

Este notebook transforma nuestras coordenadas aisladas en un tensor de entrenamiento cruzado con variables bioclimáticas continuas.

## 2. Definición de Variables Climáticas (Copernicus ERA5-Land)

Para representar el hábitat, se extrajeron dos mapas continuos (Rasters GeoTIFF) con una resolución espacial de ~11 km (EPSG:4326) abarcando desde Panamá hasta Norteamérica:
*   **Temperatura Mediana Diaria (°C):** Define las fronteras térmicas biológicas del parásito.
*   **Precipitación Acumulada Anual (mm - BIO12):** Calculada mediante la suma total temporal para evitar la dilución matemática por estacionalidad seca, capturando la humedad real de zonas selváticas vs. desérticas.

## 3. Discretización y Muestreo de Background (Pseudo-ausencias)

Los algoritmos geográficos requieren un "fondo" climático para comparar las zonas de infección contra las condiciones generales del continente. Utilizando la librería `elapid`, se aplicará el siguiente proceso:
1.  Inyección de puntos aleatorios sobre el Raster continuo para muestrear el clima base (Target = 0).
2.  Superposición de nuestras presencias reales en la base de dato para extraer la firma climática específica de los brotes (Target = 1).

A partir de esto será posible comparar el clima de los brotes reales contra el clima promedio de la zona considerada. El modelo de Máxima Entropía usado en la librería establecida buscará encontrar la regla climática que diferencía ambos grupos.

In [ ]:
import sys 
!{sys.executable} -m pip install elapid

In [4]:
import pandas as pd
import geopandas as gpd
import elapid
import os

directorio_notebooks = os.getcwd()
directorio_raiz = os.path.dirname(directorio_notebooks)

# Rutas en el directorio
ruta_raster = os.path.join(directorio_raiz, 'data', 'processed', 'Raster_Clima_GB.tif')
ruta_csv = os.path.join(directorio_raiz, 'data', 'processed', 'Dataset_GB_Mediana_Final.csv')

# Cargar casos positivos y convertirlos a formato espacial
df_casos = pd.read_csv(ruta_csv, encoding='utf-8-sig')
presencias = gpd.GeoDataFrame(
    df_casos, 
    geometry=gpd.points_from_xy(df_casos.Longitud, df_casos.Latitud), # Convierte las coordenadas en puntos 
    crs="EPSG:4326" # sistema de coordenadas eestándar de la Tierra 
)

# Muestreo de Pseudo-ausencias (35,000 puntos)
background = elapid.sample_raster(ruta_raster, count=35000)

# Usamos elapid.annotate para extraer los valores de los píxeles
x_presencias = elapid.annotate(presencias, ruta_raster) # toma las coordenadas espaciales y cruza cada punto con el mapa 
x_presencias['Target'] = 1   # los casos confirmados en el dataset
x_presencias = x_presencias[['b1', 'b2', 'Target']] # datos extraídos 

x_background = elapid.annotate(background, ruta_raster)
x_background['Target'] = 0   # ya que estamos trabajando con el background
x_background = x_background[['b1', 'b2', 'Target']]

# unir manteniendo la misma estructura 
df_entrenamiento = pd.concat([x_presencias, x_background], ignore_index=True)
df_entrenamiento = df_entrenamiento.rename(columns={
    'b1': 'Temperatura_Mediana_C', # b1, b2 son los nombres por default en la extracción
    'b2': 'Precipitacion_Anual_mm' #renombramos para mantener claridad 
})
# Filtramos solo las columnas que le interesan al modelo predictivo, eliminando puntos que caen en puntos oceánicos.
df_entrenamiento = df_entrenamiento.dropna()

# Guardamos el dataset final
df_entrenamiento.to_csv('Dataset_MaxEnt_Entrenamiento.csv', index=False)
print(f"Total de muestras para entrenar {len(df_entrenamiento)}.")


Raster:   0%|                              | 0/1 [00:00<?, ?it/s]

Sample:   0%|                              | 0/3241 [00:00<?, ?it/s]

Raster:   0%|                              | 0/1 [00:00<?, ?it/s]

Sample:   0%|                              | 0/35000 [00:00<?, ?it/s]

Total de muestras para entrenar 15079.
